# Tutorial 10: Double pendulum I

In this tutorial, we will implement some common controllers. First, we will use our knowledge of the kinematics of the double pendulum. The last controllers will also exploit the dynamics of the system. 

**Pre-requisites**

- Knowledge of kinematic and dynamic modeling of robots.
- Notions of joint-space and task-space control.
- Basics of quadratic programming.
- Understanding of common controllers.

**Goals**

Gaining a solid understanding of the complexities of trajectory stabilization.

This notebook is organized as follows:

- 1. Introduction: Double pendulum dynamics
- 2. Joint-space control
    - 2.1 Setpoint control
    - 2.2 Trajectory tracking

## 1. Introduction: Double pendulum dynamics

<img src="double_pendulum_coords.png" width="50%" align="" />

### System Description

The double pendulum has the following system parameters:

- masses $m_1$ and $m_2$
- lengths $l_1$ and $l_2$
- center of masses $r_1$ and $r_2$
- inertias $I_1$ and $I_2$

Additionally gravity $g$ acts on the masses.

In [ ]:
# imports
import numpy as np
import pandas as pd

from double_pendulum.model.plant import DoublePendulumPlant
from double_pendulum.simulation.simulation import Simulator
from double_pendulum.utils.plotting import plot_timeseries

%matplotlib inline
from IPython.display import HTML
import matplotlib as mpl
mpl.rcParams['animation.writer'] = "pillow"
import matplotlib.pyplot as plt

#from matplotlib import animation

In [ ]:
mass = [0.608, 0.630]
length = [0.3, 0.2]
com = [0.275, 0.166]
inertia = [0.05472, 0.02522]
damping = [0.0, 0.0]
torque_limit = [np.inf, np.inf]
gravity = 9.81

### Manipulator equation

The manipulator equation describes the motion of the double pendulum

$$ M(q)\ddot{q} + C(q, \dot{q})\dot{q} + G(q) = \tau $$

with the vectors

$$ q = (q_1, q_2)^T, x = (q, \dot{q})^T, \tau = (\tau_1, \tau_2)^T $$

The mass matrix is given by (with $s_1 = \sin(q_1), c_1 = \cos(q_1), \ldots$):
$$
  M =
    \left[ {\begin{array}{cc}
          I_1 + I_2 + l_1^2m_2 + 2l_1m_2r_2c_2 + g_r^2I_r + I_r &   I_2 + l_1m_2r_2c_2 \\
          I_2 + l_1m_2r_2c_2 &   I_2 \\
    \end{array}} \right]
$$

the Coriolis Matrix by:
$$
    C =
    \left[\begin{matrix}
    - 2 \dot{q}_2 l_{1} m_{2} r_{2} s_2 & - \dot{q}_2 l_{1} m_{2} r_{2} s_2\\
    \dot{q}_1 l_{1} m_{2} r_{2} s_2 &   0
    \end{matrix}\right]
$$

and the gravity vector by:
$$
    G =
    \left[\begin{matrix}
    - g m_{1} r_{1} s_1 - g m_{2} \left(l_{1}
s_1 + r_{2} s_{1+2} \right)\\
    - g m_{2} r_{2} s_{1+2}
    \end{matrix}\right]
$$

In [ ]:
dt = 0.01
t_final = 5.0
x0 = [np.pi / 2.0, -np.pi / 2.0, 0.0, 0.0]

plant = DoublePendulumPlant(
    mass=mass,
    length=length,
    com=com,
    gravity=gravity,
    inertia=inertia,
    damping=damping,
    torque_limit=torque_limit,
)

sim = Simulator(plant=plant)

T, X, U, anim = sim.simulate_and_animate(
    t0=0.0,
    x0=x0,
    tf=t_final,
    dt=dt,
    controller=None,
    integrator="runge_kutta",
)

html = HTML(anim.to_jshtml())
display(html)
plt.close()

In [ ]:
plot_timeseries(
    T,
    X,
    U,
    pos_y_lines=[0.0, np.pi],
    tau_y_lines=[-torque_limit[1], torque_limit[1]],
)

### 2.1 Setpoint control

### PID Control

Proportional-Integral-Derivative Control
$$
  \tau(x) = K_p (q_d - q) + K_i\int_0^t (q_d - q(t')) \text{d}t' - K_d\dot{q}
$$

In [ ]:
from double_pendulum.controller.pid.point_pid_controller import PointPIDController

In [ ]:
x0 = [0.0, 0.0, 0.0, 0.0]                                                                                                                                                              
goal = [np.pi, 0.0, 0.0, 0.0] 

Kp = 5.0
Ki = 0.1
Kd = 0.8

pid_controller = PointPIDController(torque_limit=torque_limit, dt=dt)
pid_controller.set_parameters(Kp=Kp, Ki=Ki, Kd=Kd)
pid_controller.set_goal(goal)
pid_controller.init()

In [ ]:
sim.reset()
T, X, U, anim = sim.simulate_and_animate(
    t0=0.0,
    x0=x0,
    tf=t_final,
    dt=dt,
    controller=pid_controller,
    integrator="runge_kutta",
)

html = HTML(anim.to_jshtml())
display(html)
plt.close()

In [ ]:
plot_timeseries(
    T,
    X,
    U,
    pos_y_lines=[0.0, np.pi],
    tau_y_lines=[-torque_limit[1], torque_limit[1]],
)

## Gravity Compensation

Counterbalance external forces (gravity)

$$
\tau(x) = G(q)
$$

Plug into EOM: $M(q)\ddot{q} + C(q, \dot{q})\dot{q} = 0$ $\rightarrow$ System without forces

In [ ]:
from double_pendulum.controller.gravity_compensation.gravity_compensation_controller import GravityCompensationController

In [ ]:
gravcomp_controller = GravityCompensationController(
    mass=[0.8*mass[0], mass[1]],
    length=length,
    com=com,
    damping=damping,
    gravity=gravity,
    inertia=inertia,
    torque_limit=torque_limit,
)

In [ ]:
x0 = [np.pi / 2.0, np.pi / 4.0, 0.0, 0.0]

sim.reset()
T, X, U, anim = sim.simulate_and_animate(
    t0=0.0,
    x0=x0,
    tf=t_final,
    dt=dt,
    controller=gravcomp_controller,
    integrator="runge_kutta",
)

html = HTML(anim.to_jshtml())
display(html)
plt.close()

In [ ]:
gravcomp_controller = GravityCompensationController(
    mass=[1.2*mass[0], mass[1]],
    length=length,
    com=com,
    damping=damping,
    gravity=gravity,
    inertia=inertia,
    torque_limit=torque_limit,
)

In [ ]:
x0 = [np.pi / 2.0, np.pi / 4.0, 0.0, 0.0]

sim.reset()
T, X, U, anim = sim.simulate_and_animate(
    t0=0.0,
    x0=x0,
    tf=t_final,
    dt=dt,
    controller=gravcomp_controller,
    integrator="runge_kutta",
)

html = HTML(anim.to_jshtml())
display(html)
plt.close()

In [ ]:
plot_timeseries(
    T,
    X,
    U,
    pos_y_lines=[0.0, np.pi],
    tau_y_lines=[-torque_limit[1], torque_limit[1]],
)

### Think-Pair-Share (5 min)

- Try different initial states (with and without velocity, etc.)
- Try modifying the instance of GravityCompensationController. Slightly change masses, dimensions, etc. How does the controller's behavior change?

### Gravity Compensation + PID

Combine the gravity compensation controller with the PID controller

$$
\tau(x) = G(q) + K_p (q_d - q) + K_i\int_0^t (q_d - q(t')) - K_d\dot{q}
$$

With this controller, we aim to add to the gravity compensation controller so that it drives our system to the goal.

In [ ]:
from double_pendulum.controller.gravity_compensation.PID_gravity_compensation_controller import PIDGravityCompensationController

In [ ]:

### Type here!

Kp = 1.0
Ki = 0.0
Kd = 1.0

###

gravcompPID_controller = PIDGravityCompensationController(
    mass=mass,
    length=length,
    com=com,
    damping=damping,
    gravity=gravity,
    inertia=inertia,
    torque_limit=torque_limit,
    dt=dt,
)
gravcompPID_controller.set_parameters(Kp=Kp, Ki=Ki, Kd=Kd)
gravcompPID_controller.set_goal(goal)

In [ ]:
x0 = [0.0, 0.0, 0.0, 0.0]

sim.reset()
sim.set_desired_state(goal)

gravcompPID_controller.init()
T, X, U, anim = sim.simulate_and_animate(
    t0=0.0,
    x0=x0,
    tf=t_final,
    dt=dt,
    controller=gravcompPID_controller,
    integrator="runge_kutta",
)

html = HTML(anim.to_jshtml())
display(html)
plt.close()

## Think-Pair-Share (5 min)

- Tune the Gravity Compensation + PID controller to get the double pendulum to swing up.
- Compare the torque with that obtained with the PID controller. How are they different? Why?

In [ ]:
plot_timeseries(
    T,
    X,
    U,
    pos_y_lines=[0.0, np.pi],
    tau_y_lines=[-torque_limit[1], torque_limit[1]],
)

### 2.2 Trajectory Tracking

## Inverse Dynamics (Feedback Linearization)

Compute the needed torque with the system inverse dynamics

$$ \tau(q, \dot{q}, \ddot{q}) = M(q)\ddot{q} + C(q, \dot{q})\dot{q} + G(q) $$

Where $\ddot{q}$ are the desired accelerations and $q$ and $\dot{q}$ are the current positions and velocities, respectively.

In [ ]:
from double_pendulum.controller.inverse_dynamics.inverse_dynamics_ff_controller import FeedforwardController

In [ ]:
# simulation parameters
dt = 0.001
t_final = 5.0
integrator = "runge_kutta"
x0 = [0.0, 0.0, 0.0, 0.0]

csv_path = "trajectory_modelfree.csv"

data = pd.read_csv(csv_path)
time_traj = np.asarray(data["time"])
pos1_traj = np.asarray(data["pos1"])
pos2_traj = np.asarray(data["pos2"])
vel1_traj = np.asarray(data["vel1"])
vel2_traj = np.asarray(data["vel2"])
acc1_traj = np.asarray(data["acc1"])
acc2_traj = np.asarray(data["acc2"])

T_des = time_traj.T
X_des = np.asarray([pos1_traj, pos2_traj, vel1_traj, vel2_traj]).T
ACC_des = np.asarray([acc1_traj, acc2_traj]).T

ff_controller = FeedforwardController(
    mass=mass,
    # mass=[0.9*m for m in mass],
    length=length,
    com=com,
    damping=damping,
    gravity=gravity,
    inertia=inertia,
    torque_limit=torque_limit,
    csv_path=csv_path,
)

In [ ]:
sim.reset()
sim.set_desired_traj(T_des, X_des)

ff_controller.init()
T, X, U, anim = sim.simulate_and_animate(
    t0=0.0,
    x0=x0,
    tf=t_final,
    dt=dt,
    controller=ff_controller,
    integrator="runge_kutta",
    plot_inittraj=True,
)

html = HTML(anim.to_jshtml())
display(html)
plt.close()

In [ ]:
acc = np.diff(np.asarray(X).T[2:, :]).T / dt
acc = np.insert(acc, 0, [0, 0], axis=0)

plot_timeseries(
    T,
    X,
    U,
    acc,
    plot_acc=True,
    plot_energy=False,
    pos_y_lines=[0.0, np.pi],
    tau_y_lines=[-torque_limit[0], torque_limit[0]],
    T_des=T_des,
    X_des=X_des,
    ACC_des=ACC_des,
)

## Think-Pair-Share (5 min)

- Why does the pendulum fall after reaching the goal position?
- Try modifying any of the values in lines 23-29 of the cell where the FeedforwardController is instantiated. How does the response of the system change?

## Trajectory-following PID Control

Proportional-Integral-Derivative Control for following a trajectory
$$
\tau(t, x) = K_p q_e(t) + K_i \int_0^t q_e(t') \text{d}t' + K_d \dot{q}_e(t) \\
q_e(t) = q_d(t) - q(t)
$$

In [ ]:
from double_pendulum.controller.pid.trajectory_pid_controller import TrajPIDController

In [ ]:
# trajectory
#csv_path = "trajectory_modelfree.csv"
csv_path = "trajectory_modelbased_dfki.csv"

data = pd.read_csv(csv_path)
time_traj = np.asarray(data["time"])
pos1_traj = np.asarray(data["pos1"])
pos2_traj = np.asarray(data["pos2"])
vel1_traj = np.asarray(data["vel1"])
vel2_traj = np.asarray(data["vel2"])
acc1_traj = np.asarray(data["acc1"])
acc2_traj = np.asarray(data["acc2"])

T_des = time_traj
X_des = np.asarray([pos1_traj, pos2_traj, vel1_traj, vel2_traj]).T
ACC_des = np.asarray([acc1_traj, acc2_traj]).T

# controller parameters
Kp = 10.0
Ki = 0.0
Kd = 10.0

trajPID_controller = TrajPIDController(csv_path=csv_path, dt=dt, torque_limit=torque_limit)
trajPID_controller.set_parameters(Kp=Kp, Ki=Ki, Kd=Kd)

In [ ]:
sim.reset()
sim.set_desired_traj(T_des, X_des)

trajPID_controller.init()
T, X, U, anim = sim.simulate_and_animate(
    t0=0.0,
    x0=x0,
    tf=t_final,
    dt=dt,
    controller=trajPID_controller,
    integrator="runge_kutta",
    plot_inittraj=True,
)

html = HTML(anim.to_jshtml())
display(html)
plt.close()

In [ ]:
acc = np.diff(np.asarray(X).T[2:, :]).T / dt
acc = np.insert(acc, 0, [0, 0], axis=0)

plot_timeseries(
    T,
    X,
    U,
    acc,
    plot_acc=True,
    plot_energy=False,
    pos_y_lines=[0.0, np.pi],
    tau_y_lines=[-torque_limit[0], torque_limit[0]],
    T_des=T_des,
    X_des=X_des,
    ACC_des=ACC_des,
)

## Computed Torque

A combination of feedforward and PID

$$
\tau(t, x) = M(q(t))\left( \ddot{q}_d(t) + K_p q_e(t) + K_i \int_0^t q_e(t') \text{d}t' + K_d \dot{q}_e(t) \right) + C(q(t), \dot{q}(t))\dot{q}(t) + G(q(t))
$$

- PID control is plugged in on the acceleration level. PID control signal is added to the desired acceleration $\ddot{q}_d$

In [ ]:
from double_pendulum.controller.inverse_dynamics.computed_torque_controller import ComputedTorqueController

In [ ]:
### Type here!

Kp = 1.0
Ki = 0.0
Kd = 1.0

###

csv_path = "trajectory_modelbased_dfki.csv"

data = pd.read_csv(csv_path)
time_traj = np.asarray(data["time"])
pos1_traj = np.asarray(data["pos1"])
pos2_traj = np.asarray(data["pos2"])
vel1_traj = np.asarray(data["vel1"])
vel2_traj = np.asarray(data["vel2"])
acc1_traj = np.asarray(data["acc1"])
acc2_traj = np.asarray(data["acc2"])

T_des = time_traj.T
X_des = np.asarray([pos1_traj, pos2_traj, vel1_traj, vel2_traj]).T
ACC_des = np.asarray([acc1_traj, acc2_traj]).T

ctg_controller = ComputedTorqueController(
    mass=mass,
    length=length,
    com=com,
    damping=damping,
    gravity=gravity,
    inertia=inertia,
    torque_limit=torque_limit,
    dt=dt,
    csv_path=csv_path,
)

ctg_controller.set_parameters(Kp=Kp, Ki=Ki, Kd=Kd)

In [ ]:
sim.reset()
sim.set_desired_traj(T_des, X_des)

ctg_controller.init()
T, X, U, anim = sim.simulate_and_animate(
    t0=0.0,
    x0=x0,
    tf=t_final,
    dt=dt,
    controller=ctg_controller,
    integrator="runge_kutta",
    plot_inittraj=True,
)

html = HTML(anim.to_jshtml())
display(html)
plt.close()

In [ ]:
acc = np.diff(np.asarray(X).T[2:, :]).T / dt
acc = np.insert(acc, 0, [0, 0], axis=0)

plot_timeseries(
    T,
    X,
    U,
    acc,
    plot_acc=True,
    plot_energy=False,
    pos_y_lines=[0.0, np.pi],
    tau_y_lines=[-torque_limit[0], torque_limit[0]],
    T_des=T_des,
    X_des=X_des,
    ACC_des=ACC_des,
)

### Think-Pair-Share (5 min)
- Tune the Computed torque controller.
- Compare the torque with that obtained with the PID controller. How are they different?

## Swing-up on the real system

As a final step, we can apply the computed-torque controller to swing-up on the real system:

In [ ]:
import os
import time
try:
    from pycloudpendulum import Client
    USE_LOCAL = False
except:
    from cloud_pendulum_local import Client
    USE_LOCAL = True

Add your user token here. The following cell works if you have your token in a file called `.env` in your home directory on the CloudPendulum server. (Simply open a terminal > `nano ~/.env` > Paste your token > `CTRL + X`, then `Y`, then `Enter`. Possibly restart the kernel.) 

Alternatively, you can simply pass it as `user_token = "user_token_string"`, but please do not commit the token to public repositories.

If you run the notebook on a local setup, no token is needed.

In [ ]:
if USE_LOCAL:
    user_token = "dummy_token"
else:
    with open(os.path.expanduser("~/.env"), 'r') as file:
        user_token = file.read().strip('\n')

client = Client()

print("="*30)
print("Starting CloudPendulum")
print(f"{client.get_user_info(user_token).number_of_attempts} attempts left")
print("="*30)

In [ ]:
# model of the real cloud pendulum
mass = [ 0.10548177618443695, 0.07619744360415454 ]
length=[0.05, 0.05]
com = [ 0.05, 0.03670036749567022 ]
damping = [ 7.634058385430087e-12,  0.0005106535523065844 ]
gravity=9.81
inertia = [ 0.00046166221821039165, 0.00023702395072092597 ]
torque_limit=[0.14, 0.14]
dt=0.01

csv_path = "trajectory_modelbased.csv"

ctg_controller = ComputedTorqueController(
    mass=mass,
    length=length,
    com=com,
    damping=damping,
    gravity=gravity,
    inertia=inertia,
    torque_limit=torque_limit,
    dt=dt,
    csv_path=csv_path,
)

Since we now use the parameters of the physical system, you need to retune your PID controller:

In [ ]:
plan = pd.read_csv(csv_path)
x0 = [0.0, 0.0, 0.0, 0.0]
T_des = plan['time'].iloc[-1]
t_final=T_des
X_des = plan[["pos1", "pos2", "vel1", "vel2"]]
U_des = plan[["tau0", "tau1"]]

# Tune your PID gains here
Kp = 70.0
Ki = 20.0
Kd = 8.0
# Tune your PID gains here 

ctg_controller.set_parameters(Kp=Kp, Ki=Ki, Kd=Kd)

ctg_controller.init()
p2 = DoublePendulumPlant(mass=mass, length=length, com=com, gravity=gravity, inertia=inertia, damping=damping, torque_limit=torque_limit,)

s2 = Simulator(plant=p2)
s2.reset()
s2.set_desired_traj(T_des, X_des)

T, X, U = s2.simulate(
    t0=0.0,
    x0=x0,
    tf=t_final,
    dt=dt,
    controller=ctg_controller,
    integrator="runge_kutta",
)

plt.figure(figsize=(14, 4))
for i, (traj, name) in enumerate(zip(np.array(X).T, [r"$\theta_1$", r"$\theta_2$", r"$\dot \theta_1$", r"$\dot \theta_2$"])):
    plt.subplot(2, 2, i+1)
    plt.plot(T, traj)
    plt.ylabel(name)
    # plt.xlim(0, 10)
    plt.grid()

for i, pdname in enumerate(["pos1", "pos2", "vel1", "vel2"]):
    plt.subplot(2, 2, i+1)
    plt.plot(T, plan[pdname], "--")
    plt.legend(["real", "planned"])

plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 2))
plt.subplot(1, 2, 1)
plt.plot(T, np.asarray(U)[:, 0])
plt.plot(T, plan["tau0"])
plt.legend(["real", "planned"])
plt.ylabel(r"$\tau_0$")
plt.grid()

plt.subplot(1, 2, 2)
plt.plot(T, np.asarray(U)[:, 1])
plt.plot(T, plan["tau1"])
plt.legend(["real", "planned"])
plt.ylabel(r"$\tau_1$")
plt.grid()

plt.tight_layout()


Now we can start the experiment:

In [ ]:
experiment_time = 5
preparation_time = 5

x0 = [np.pi, 0]
n = int(experiment_time / dt)

def wait_for_control_loop_end(time_to_pass):
        """Delay ending a while loop so that it loops at a desired sampling time dt."""
        if time_to_pass <= 0.0:
            return

        start = time.time()
        
        time.sleep(time_to_pass * 0.7) # sleep
        while time.time() - start < time_to_pass: # busy waiting
            pass

In [ ]:
session_token, url = client.start_experiment(user_token, "DoublePendulum", experiment_time, preparation_time, record=True)

client.set_impedance_controller_params([0.0], [0.0], session_token)

tau_max = 0.13

ctg_controller.init()
meas_time_vec = np.zeros(n)
meas_x = np.zeros((n, 4))
meas_u = np.zeros((n, 2))
# pred_x = np.zeros((n, 4))
des_u = np.zeros((n, 2))

i = 0
tau = 0.0
t_start = time.time()

while i < n:
    meas_time_vec[i] = time.time() - t_start

    # Measure state
    measured_x = np.array( client.get_position(session_token) + client.get_velocity(session_token))
    measured_torque = client.get_torque(session_token)

    tau = ctg_controller.get_control_output(measured_x)
    tau = np.clip(tau, -tau_max, tau_max)
    client.set_torque(list(tau), session_token)
    
    # Record data
    meas_x[i] = measured_x
    meas_u[i] = measured_torque
    des_u[i] = tau
    
    i += 1
    sleep_duration = t_start + i * dt - time.time()
    wait_for_control_loop_end(sleep_duration)

try:
    video_url = client.stop_experiment(session_token)

    if not USE_LOCAL:
        # Show video
        from misc import download_video
        video_path = download_video(video_url)
        from IPython.display import Video
        display(Video(video_path, embed=True))
except:
    pass


In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(3, 1, 1)
plt.plot(meas_time_vec, meas_x[:, 0], "b", label=r"$q_1$")
plt.plot(meas_time_vec, meas_x[:, 1], "r", label=r"$q_2$")
plt.plot(meas_time_vec, plan["pos1"][:500], "b--", label=r"Planned $q_1$")
plt.plot(meas_time_vec, plan["pos2"][:500], "r--", label=r"Planned $q_2$")

plt.subplot(3, 1, 2)
plt.plot(meas_time_vec, meas_x[:, 2], "b", label=r"$\dot q_1$")
plt.plot(meas_time_vec, meas_x[:, 3], "r", label=r"$\dot q_2$")
plt.plot(meas_time_vec, plan["vel1"][:500], "b--", label=r"Planned $\dot q_1$")
plt.plot(meas_time_vec, plan["vel2"][:500], "r--", label=r"Planned $\dot q_2$")

plt.subplot(3, 1, 3)
plt.plot(meas_time_vec, meas_u[:, 0], "b", label=r"$\tau_1$")
plt.plot(meas_time_vec, meas_u[:, 1], "r", label=r"$\tau_2$")
plt.plot(meas_time_vec, plan["tau0"][:500], "b--", label=r"Planned $\tau_1$")
plt.plot(meas_time_vec, plan["tau1"][:500], "r--", label=r"Planned $\tau_2$")


for i, name in enumerate(["angle (rad)", "ang. vel. (rad/s)", "input (Nm)"]):
    plt.subplot(3, 1, i+1)
    plt.legend(loc="right")
    plt.grid()
    plt.ylabel(name)
plt.tight_layout()